# 15 — Integrated SDK wheel verification (candidate)

**Kaggle verification of the integrated wheel.** This notebook installs the single integrated wheel `stokastisk-0.1.0-2-cp313-cp313-linux_x86_64.whl` (Stokastisk 0.1.0 + bundled vLLM 0.25.1, Triton attention only, SM75) into a fresh session and verifies it; it does not repeat the TP/PP/replica benchmarks (notebooks 13 and 14 verified those on the legacy two-wheel bundle). Run it on Kaggle with Accelerator **GPU T4 x2**, Internet ON, Python 3.13. Statuses: PASS / FAIL / ERROR / WARNING / SKIPPED / NOT_RUN; an unsupported feature is never reported as PASS. Outputs: `/kaggle/working/15-stokastisk-candidate-acceptance.zip`.

Verifies: Python 3.13, PyTorch 2.11.0+cu130, Triton 3.6.0, CUDA 13.0, two T4 (SM75); wheel SHA256; installation without touching protected packages; versions; public API; `stokastisk.compat`; `stokastisk.serving` (real `vllm serve`, HTTP, streaming, metrics); `LLM.chat`; Qwen2.5-0.5B-Instruct FP16 inference; `TRITON_ATTN`; xgrammar structured output; bfloat16 detection (rejected, never run); clean shutdown and GPU memory release; no Flash library; honest `pip check`.

## 1. Locate the integrated release bundle

In [1]:
import glob, hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor
import hashlib, json, os, pathlib, shutil, sys, time, urllib.request, zipfile
from concurrent.futures import ThreadPoolExecutor

HF_REPO = "waqasm86/stokastisk"
HF_BASE = f"https://huggingface.co/{HF_REPO}/resolve"


def _open(url, headers=None):
    return urllib.request.urlopen(urllib.request.Request(url, headers=headers or {"User-Agent": "stokastisk-fetch/1"}), timeout=60)


def _range_ok(url):
    if not url.startswith("http"):
        return None
    with _open(url, {"Range": "bytes=0-0", "User-Agent": "stokastisk-fetch/1"}) as r:
        cr = r.headers.get("Content-Range")                      # "bytes 0-0/37080766"
        return int(cr.rsplit("/", 1)[1]) if r.status == 206 and cr else None


def _download(url, dest, workers=1, retries=3):
    """Download to `<dest>.part`, check free disk space and the advertised length, then replace `dest` atomically; the partial file is removed on any failure."""
    dest = pathlib.Path(dest); part = dest.with_name(dest.name + ".part")
    part.unlink(missing_ok=True)
    try:
        total = _range_ok(url)
        if total and shutil.disk_usage(dest.parent).free < total + (64 << 20):
            raise RuntimeError(f"not enough free disk space in {dest.parent} for {total / 1e6:.0f} MB")
        _download_to(url, part, workers, retries)
        if total and part.stat().st_size != total:
            raise RuntimeError(f"incomplete download: {part.stat().st_size} of {total} bytes")
        os.replace(part, dest)
    except BaseException:
        part.unlink(missing_ok=True)
        raise


def _download_to(url, dest, workers=1, retries=3):
    dest = pathlib.Path(dest); total = _range_ok(url)
    t0 = time.time()
    if total and total > (4 << 20):
        chunk = -(-total // workers); parts = [(i * chunk, min(total, (i + 1) * chunk) - 1) for i in range(workers) if i * chunk < total]
        with open(dest, "wb") as f:
            f.truncate(total)

        def part(rng):
            a, b = rng
            for attempt in range(retries):
                try:
                    with _open(url, {"Range": f"bytes={a}-{b}", "User-Agent": "stokastisk-fetch/1"}) as r, open(dest, "r+b") as f:
                        f.seek(a); shutil.copyfileobj(r, f, 1 << 20)
                    return
                except Exception:
                    if attempt == retries - 1: raise
                    time.sleep(2 ** attempt)
        with ThreadPoolExecutor(len(parts)) as ex:
            list(ex.map(part, parts))
        mode = f"{len(parts)} parallel ranges"
    else:
        for attempt in range(retries):
            try:
                with _open(url) as r, open(dest, "wb") as f:
                    shutil.copyfileobj(r, f, 1 << 20)
                break
            except Exception:
                if attempt == retries - 1: raise
                time.sleep(2 ** attempt)
        mode = "single stream"
    n = dest.stat().st_size
    print(f"downloaded {n / 1e6:.1f} MB in {time.time() - t0:.1f}s ({n / 1e6 / max(time.time() - t0, 1e-6):.1f} MB/s, {mode})")


def _sha256(path):
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for c in iter(lambda: f.read(1 << 20), b""):
            h.update(c)
    return h.hexdigest()


def fetch_release(release, dest="/tmp/stokastisk-release", base=None, revision="main", workers=1):
    """Return the verified, extracted bundle directory (…/<release>). `base` overrides the Hub URL prefix (tests use file://)."""
    base = (base or HF_BASE).rstrip("/")
    root = f"{base}/{revision}/releases/{release}" if base.startswith("http") else f"{base}/releases/{release}"
    dest = pathlib.Path(dest); dest.mkdir(parents=True, exist_ok=True)
    zname = f"{release}-kaggle-input.zip"; zpath = dest / zname
    if zpath.exists(): zpath.unlink()
    with _open(f"{root}/{zname}.sha256") as r:
        expected = r.read().decode().split()[0]
    _download(f"{root}/{zname}", zpath, workers=workers)
    got = _sha256(zpath)
    if got != expected:
        zpath.unlink(); raise RuntimeError(f"SHA256 mismatch for {zname}: expected {expected}, got {got}")
    out = dest / release
    if out.exists(): shutil.rmtree(out)
    with zipfile.ZipFile(zpath) as z:
        assert z.testzip() is None
        for n in z.namelist():
            if n.startswith("/") or ".." in pathlib.PurePosixPath(n).parts or not n.startswith(release + "/"):
                raise RuntimeError(f"unsafe or foreign path in archive: {n}")
        z.extractall(dest)
    m = json.loads((out / "manifests" / "RELEASE-MANIFEST.json").read_text())
    if m.get("name") != release:
        raise RuntimeError(f"manifest name {m.get('name')!r} != {release!r}")
    print(f"release {release}: SHA256 {got[:16]}... verified, extracted to {out}")
    return out

# Only the bundle directory named RELEASE whose RELEASE-MANIFEST.json carries the same name is accepted (never an older rc).
RELEASE = "stokastisk-0.1.0-integrated-kaggle-t4-sm75"
cands = [pathlib.Path(p).parent.parent for p in glob.glob("/kaggle/input/**/manifests/SHA256SUMS", recursive=True)]
if os.environ.get("STOKASTISK_BUNDLE"): cands = [pathlib.Path(os.environ["STOKASTISK_BUNDLE"])]
match = [c for c in cands if c.name == RELEASE]
if not match:
    print(f"{RELEASE} is not attached (attached: {sorted(str(c) for c in cands)}); downloading from Hugging Face (needs Internet ON and a published release) ...")
    match = [fetch_release(RELEASE)]
assert len({(c / "manifests" / "SHA256SUMS").read_bytes() for c in match}) == 1, f"several DIFFERENT copies of {RELEASE}: {match}"
BUNDLE = match[0]
_m = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())
assert _m.get("name") == RELEASE, f"manifest name {_m.get('name')!r} != expected {RELEASE!r}"
SCRIPTS = BUNDLE / "scripts" / "kaggle"
os.environ["STOKASTISK_OUT"] = "/kaggle/working/stokastisk-acceptance"
print("release:", RELEASE, "| bundle:", BUNDLE, "| manifest sha256:", hashlib.sha256((BUNDLE / "manifests" / "SHA256SUMS").read_bytes()).hexdigest())

stokastisk-0.1.0-integrated-kaggle-t4-sm75 is not attached (attached: []); downloading from Hugging Face (needs Internet ON and a published release) ...
downloaded 37.1 MB in 0.9s (41.7 MB/s, 1 parallel ranges)
release stokastisk-0.1.0-integrated-kaggle-t4-sm75: SHA256 f28ccdf230fb80f3... verified, extracted to /tmp/stokastisk-release/stokastisk-0.1.0-integrated-kaggle-t4-sm75
release: stokastisk-0.1.0-integrated-kaggle-t4-sm75 | bundle: /tmp/stokastisk-release/stokastisk-0.1.0-integrated-kaggle-t4-sm75 | manifest sha256: dce0e2987d15d21fb88f22610ecbca1eb6877791dc9fc094186a1562e5ce3e6b


## 2. Provenance: scripts and the wheel must match `manifests/SHA256SUMS`

In [2]:
import hashlib
sha = lambda p: hashlib.sha256(open(p, "rb").read()).hexdigest()
for w in sorted((BUNDLE / "wheels").glob("*.whl")): print(f"{sha(w)}  {w.name}")
expected = {l.split(None, 1)[1].strip(): l.split()[0] for l in (BUNDLE / "manifests" / "SHA256SUMS").read_text().splitlines()}
for rel in [f"scripts/kaggle/{s.name}" for s in SCRIPTS.glob("*.py")] + [f"wheels/{w.name}" for w in (BUNDLE / "wheels").glob("*.whl")]:
    assert expected.get(rel) == sha(BUNDLE / rel), f"{rel} does not match manifests/SHA256SUMS"
print("all executed scripts and wheels match manifests/SHA256SUMS")

7e4b700929ec7f3489570e93252ef5a49da2d1682c111c5feefb55efead84369  stokastisk-0.1.0-2-cp313-cp313-linux_x86_64.whl
all executed scripts and wheels match manifests/SHA256SUMS


## 3. Install the integrated wheel into this fresh session (`--no-deps`, hash-verified pins, protected packages untouched)

In [3]:
import subprocess, sys, os, json
r = subprocess.run([sys.executable, str(SCRIPTS / "02_install_integrated.py"), str(BUNDLE)],
                   env=dict(os.environ, STOKASTISK_OUT="/kaggle/working/stokastisk-acceptance/prereq-install"))
assert r.returncode == 0, "integrated installer failed (see output above and prereq-install/02_install_integrated.json); do not continue"
EXPECTED_WHEEL_SHA256 = json.loads((BUNDLE / "manifests" / "RELEASE-MANIFEST.json").read_text())["integrated_wheel"]["sha256"]
print("integrated wheel sha256 (manifest):", EXPECTED_WHEEL_SHA256)

PASS SHA256SUMS verified before any install -- 29 files match
PASS exactly one wheel, the integrated wheel -- ['stokastisk-0.1.0-2-cp313-cp313-linux_x86_64.whl']
PASS wheel SHA256 equals RELEASE-MANIFEST.json -- 7e4b700929ec7f3489570e93252ef5a49da2d1682c111c5feefb55efead84369
PASS python 3.13 -- 3.13.16
PASS baseline protected (torch/triton exact, transformers>=5.5.3,<6, no prohibited packages) -- {'torch': '2.11.0+cu130', 'triton': '3.6.0', 'transformers': '5.20.2', 'numpy': '1.26.4', 'nvidia-cuda-nvcc': '13.0.88', 'nvidia-nvvm': '13.0.88'}
PASS PyTorch CUDA build is 13.0 -- 13.0
PASS two Tesla T4 GPUs (SM75) -- ['Tesla T4', 'Tesla T4'] [(7, 5), (7, 5)]
PASS wheel tags: cp313-cp313-linux_x86_64, build tag 2 -- ('cp313-cp313-linux_x86_64', '2')
PASS wheel has exactly the packages stokastisk + vllm (no duplicate vLLM tree) -- ['stokastisk', 'vllm']
PASS wheel registers the vllm 0.25.1 distribution (vLLM calls importlib.metadata.version('vllm'))
PASS wheel contains no FlashAttention/Flas

## 4. Stages (separate processes)

In [4]:
import subprocess, sys, os
def run_stage(script, *args, timeout=1800, log=None):
    """Run a bundled stage script with a finite timeout; returns its exit code (124 = timeout). Failures are judged in the verdict cell."""
    env = dict(os.environ, STOKASTISK_EXPECTED_WHEEL_SHA256=EXPECTED_WHEEL_SHA256, PYTHONPATH=str(SCRIPTS) + ":" + os.environ.get("PYTHONPATH", ""), VLLM_LOGGING_LEVEL="INFO")
    out = open("/kaggle/working/stokastisk-acceptance/" + (log or script + ".log"), "w") if log is not False else None
    try:
        r = subprocess.run([sys.executable, str(SCRIPTS / script), *args], cwd=SCRIPTS, env=env, timeout=timeout, stdout=out, stderr=subprocess.STDOUT if out else None)
        rc = r.returncode
    except subprocess.TimeoutExpired:
        rc = 124
    print(f"{script} {' '.join(args)} exit code: {rc}")
    return rc
os.makedirs("/kaggle/working/stokastisk-acceptance", exist_ok=True)

In [5]:
RC = {}
for mode, tmo in (("static", 900), ("engine", 1500), ("server", 1800)):
    RC["15c_" + mode] = run_stage("candidate_15_integration.py", mode, timeout=tmo, log=f"15c_{mode}.log")
RC["15c_summary"] = run_stage("candidate_15_integration.py", "summary", timeout=120, log="15c_summary.log")

candidate_15_integration.py static exit code: 1
candidate_15_integration.py engine exit code: 1
candidate_15_integration.py server exit code: 1
candidate_15_integration.py summary exit code: 1


## Package evidence → `/kaggle/working/15-stokastisk-candidate-acceptance.zip`

In [6]:
import shutil, json as _j
shutil.copy(BUNDLE / "manifests" / "SHA256SUMS", "/kaggle/working/stokastisk-acceptance/bundle-SHA256SUMS")
shutil.copy(BUNDLE / "manifests" / "RELEASE-MANIFEST.json", "/kaggle/working/stokastisk-acceptance/bundle-RELEASE-MANIFEST.json")
print(shutil.make_archive("/kaggle/working/15-stokastisk-candidate-acceptance", "zip", "/kaggle/working", "stokastisk-acceptance"))

/kaggle/working/15-stokastisk-candidate-acceptance.zip


## Verdict (fails the notebook if a REQUIRED stage failed)

All four stages are required. A WARNING (for example the documented xgrammar pip-check conflict) is reported, not hidden.

In [7]:
required = ['15c_static', '15c_engine', '15c_server', '15c_summary']
bad = {k: v for k, v in RC.items() if k in required and v != 0}
print('all stage exit codes:', RC)
assert not bad, f'FAILED required stage(s): {bad}'

all stage exit codes: {'15c_static': 1, '15c_engine': 1, '15c_server': 1, '15c_summary': 1}


AssertionError: FAILED required stage(s): {'15c_static': 1, '15c_engine': 1, '15c_server': 1, '15c_summary': 1}